In [1]:
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm
import glob

from utils import hasher

In [8]:
# # for testing purposes
# def hasher(p):
#     return p

In [3]:
'''
This script parses through collected questions and contexts. 
Assumption that context is a single image.

Input:
papername_collector/questions/*.png
papername_collector/contexts/*.png

Output:
metadata.json
annotations.csv -> which can be converted to annotations.json once populated.
'''

'\nThis script parses through collected questions and contexts. \nAssumption that context is a single image.\n\nInput:\npapername_collector/questions/*.png\npapername_collector/contexts/*.png\n\nOutput:\nmetadata.json\nannotations.csv -> which can be converted to annotations.json once populated.\n'

In [14]:
dirname = 'papertemp_UA' ## convention: papername_collector. # collector should not have _ in it.
qdir = 'screenshots'
cdir = 'screenshots'

os.listdir(dirname)

['contexts', 'questions', 'screenshots']

In [15]:
files = [os.path.basename(p) for p in glob.glob(os.path.join(dirname,qdir,'*.png'))]

contexts = [os.path.basename(p) for p in glob.glob(os.path.join(dirname,cdir,'*.png')) if os.path.basename(p)[0]=='c']
questions = pd.Series([p.split('_')[0] for p in files if p[0]=='q']).unique()  
# number of unique questions, i.e. counting parts as same q
# ensuring we only count files beginning with q, in case contexts and qs are in the same 
print(len(files),len(questions),len(contexts))

8 4 2


In [16]:
questions,contexts

(array(['q0', 'q1', 'q2', 'q3'], dtype=object), ['c1.png', 'c2.png'])

In [22]:
### Field for metadata
papername='_'.join(dirname.split('_')[:-1]) # in case paper name has _
paper_id=hasher(papername) # paper id
num_questions=len(questions)
annotator=dirname.split('_')[-1]

meta = {
    'file_name':papername,
    'paper_id':paper_id,
    'num_questions':num_questions,
    'annotator':annotator,
    'questions_relative_path':qdir,
    'contexts_relative_path':cdir    
}

op = os.path.join(dirname,'metadata.json')

with open(op,'w') as f:
    json.dump([meta],f)
    
print(f'Paper name: {papername}\nCollected/Annotated by: {annotator}')
print(f'saved in {op}')
print(meta)

Paper name: papertemp
Collected/Annotated by: UA
saved in papertemp_UA/metadata.json
{'file_name': 'papertemp', 'paper_id': 'papertemp', 'num_questions': 4, 'annotator': 'UA', 'questions_relative_path': 'screenshots', 'contexts_relative_path': 'screenshots'}


In [18]:
print(f'there are {num_questions} questions')

there are 4 questions


In [19]:
### Fields for sample annotation

sample_ids=questions # list of questions
paper_ids=[paper_id]*len(questions) # paper id
imgpaths=[] # list of lists; images for this question
contexts=[] # context per question
input_text_parsed=[] # model-parsed text containing [diag description] / [questions] / [options] 
# ^ currently empty, can populate in this script by calling the model here.

# to annotate manually:
instructions=[] # section instructions for this question
explanation=[]
final_answer=[]
final_answer_range=[]
page_number=[]
exps=[]

In [20]:
for i,q in tqdm(enumerate(questions),total=len(questions)):
    
    imgs = glob.glob(os.path.join(dirname,qdir,f'{q}*.png')) # list of all images for this q
    imgpaths.append([os.path.basename(p) for p in imgs]) # in case single question is split into multiple screenshots
    imgex = imgpaths[-1][-1]
    
    if('c' in imgex): # i.e. context exists for this question
        cfile = imgex.split('_')[-1]
        if not os.path.isfile(os.path.join(dirname,cdir,cfile)):
            print('context not found!')
            contexts.append('ERROR')
        else:
            contexts.append(cfile)
    else: contexts.append('NULL')
        
    
    input_text_parsed.append('')
    instructions.append('')
    final_answer.append('')
    final_answer_range.append('')
    page_number.append('')
    exps.append('')

100%|██████████| 4/4 [00:00<00:00, 433.91it/s]


In [21]:
anfile = pd.DataFrame(data={
    'paper_id-input':paper_ids,
    'sample_id-input':sample_ids,
    'page_number-input':page_number,
    'input_image_location-input':imgpaths,
    'section_instruction-input':instructions,
    'context-input':contexts,
    'input_text_parsed-input':input_text_parsed,
    'explanation-output':exps,
    'final_answer-output':final_answer,
    'final_answer_range-output':final_answer_range 
    
})

## adding -input and -output delimiters to make JSON conversion easier later

anfile.to_csv(os.path.join(dirname,'annotations.csv'),index=False)